## 2. Feature Mappings for Image Reconstruction

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os
import random
import hashlib
import tqdm
from tqdm import tqdm
import math
import time
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

In [ ]:
username = "gracy.garg"
# seed = 42
seed = int ( hashlib . sha256 ( username . encode () ) . hexdigest () , 16)
seed = seed % (2**32)
np.random.seed(seed)
random.seed(seed)
# print(f"Seed for {username}: {seed}")

In [ ]:
def add_username_to_plot(username="gracy.garg"):
    """
    Adds the username to the top-right corner of the current plot.
    """
    plt.text(
        0.95, 0.95, username,
        ha='right', va='top',
        transform=plt.gca().transAxes,
        fontsize=10, color='gray', alpha=0.7
    )

In [ ]:
class ReLU:
    def forward(self, z):
        self.output = np.maximum(0, z)
        return self.output

    def backward(self, grad_output):
        # Derivative is 1 where z > 0, and 0 elsewhere
        grad_input = grad_output * (self.output > 0)
        return grad_input

In [ ]:
class Tanh:
    def forward(self, z):
        self.output = np.tanh(z)
        return self.output

    def backward(self, grad_output):
        # Derivative of tanh(z) is 1 - tanh(z)^2
        grad_input = grad_output * (1 - self.output**2)
        return grad_input

In [ ]:
class Sigmoid:
    def forward(self, z):
        self.output = 1 / (1 + np.exp(-z))
        return self.output

    def backward(self, grad_output):
        # Derivative of sigmoid(z) is sigmoid(z) * (1 - sigmoid(z))
        grad_input = grad_output * self.output * (1 - self.output)
        return grad_input

In [ ]:
class Identity:
    def forward(self, z):
        self.output = z
        return self.output

    def backward(self, grad_output):
        # Derivative is 1
        return grad_output

In [ ]:
class LinearLayer:
    def __init__(self, input_width, output_width, activation_fn):
        # Forward pass data
        self.input_width = input_width
        self.output_width = output_width
        self.activation_fn = activation_fn

        # Initialize weights (W) and biases (b)
        # Use Xavier/He initialization (here, basic small random initialization)
        self.W = np.random.randn(input_width, output_width) * np.sqrt(2 / input_width)
        self.b = np.zeros((1, output_width))

        # Backward pass data
        self.x = None              # Layer input (x)
        self.layer_output = None   # Output after linear op (z = xW + b)
        self.grad_W_acc = np.zeros_like(self.W)
        self.grad_b_acc = np.zeros_like(self.b)

    def forward(self, x):
        # Store input for backward pass
        self.x = x

        # Linear operation: z = xW + b (x is batch_size x input_width)
        self.layer_output = np.dot(x, self.W) + self.b

        # Activation
        activated_output = self.activation_fn.forward(self.layer_output)
        return activated_output

    def backward(self, grad_output, accumulate=True): # <--- MODIFICATION HERE
            # 1. Backpropagate through the activation function
            grad_z = self.activation_fn.backward(grad_output)

            # 2. Calculate gradients for W, b, and input (x)
            grad_W = np.dot(self.x.T, grad_z)
            grad_b = np.sum(grad_z, axis=0, keepdims=True)
            grad_input = np.dot(grad_z, self.W.T)

            # 3. Accumulate gradients <--- MODIFICATION HERE
            if accumulate: # ONLY accumulate during the analytic backward pass
                self.grad_W_acc += grad_W
                self.grad_b_acc += grad_b

            return grad_input

    def zero_grad(self):
        """Resets the cumulative gradient to zero."""
        self.grad_W_acc.fill(0)
        self.grad_b_acc.fill(0)

    def update(self, learning_rate, grad_accumulation_steps):
        """Updates parameters using accumulated gradients and resets them."""
        # Note: Gradients accumulated over 'grad_accumulation_steps' batches.
        # Normalize by the number of samples or batches if necessary, but here
        # we'll use a simple update proportional to the accumulated gradient.

        # Normalize accumulation by steps to keep learning rate consistent across different settings
        W_update = self.grad_W_acc / grad_accumulation_steps
        b_update = self.grad_b_acc / grad_accumulation_steps

        self.W -= learning_rate * W_update
        self.b -= learning_rate * b_update

        self.zero_grad()

In [ ]:
class Loss:
    """Base class for loss functions."""
    def compute(self, y_pred, y_true):
        raise NotImplementedError

    def backward(self, y_pred, y_true):
        raise NotImplementedError

class MSELoss(Loss):
    def compute(self, y_pred, y_true):
        """Mean Squared Error: L = 1/N * sum((y_pred - y_true)^2)"""
        return np.mean((y_pred - y_true)**2)

    def backward(self, y_pred, y_true):
        """Derivative: dL/dy_pred = 2/N * (y_pred - y_true)"""
        N = y_pred.shape[0] # Batch size
        return 2 * (y_pred - y_true) / N

class BCELoss(Loss):
    def compute(self, y_pred, y_true, epsilon=1e-12):
        """Binary Cross Entropy: L = -1/N * sum(y_true*log(y_pred) + (1-y_true)*log(1-y_pred))"""
        # Clip predictions to prevent log(0)
        y_pred = np.clip(y_pred, epsilon, 1. - epsilon)
        loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        return loss

    def backward(self, y_pred, y_true, epsilon=1e-12):
        """Derivative: dL/dy_pred = -1/N * (y_true/y_pred - (1-y_true)/(1-y_pred))"""
        y_pred = np.clip(y_pred, epsilon, 1. - epsilon)
        N = y_pred.shape[0]
        grad = (y_pred - y_true) / (y_pred * (1 - y_pred))
        return grad  # The N factor is already in the BCE loss derivation for a batch mean

In [ ]:
class Model:
    def __init__(self, layers, loss_fn_type):
        self.layers = layers
        self.loss_fn = self._get_loss_function(loss_fn_type)

    def _get_loss_function(self, loss_fn_type):
        """Initializes the loss function based on the type string."""
        if loss_fn_type.lower() == 'mse':
            return MSELoss()
        elif loss_fn_type.lower() == 'bce':
            return BCELoss()
        else:
            raise ValueError(f"Unknown loss function type: {loss_fn_type}")

    def forward(self, x):
        """Performs a forward pass through all layers."""
        out = x
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, grad_output, accumulate=True): # <--- MODIFICATION HERE
        """Performs a backward pass (backpropagation)."""
        grad = grad_output
        # Backpropagate in reverse order
        for layer in reversed(self.layers):
            grad = layer.backward(grad, accumulate=True)
        # Returns the gradient w.r.t the model input, though not strictly needed here

    def train(self, x, y):
        """
        Performs forward pass, computes loss, backpropagates, and accumulates gradients.
        (1.2.3 train(x,y))
        """

        y_pred = self.forward(x)                       # 1. Forward Pass
        loss = self.loss_fn.compute(y_pred, y)         # 2. Compute Loss
        grad_loss = self.loss_fn.backward(y_pred, y)   # 3. Calculate initial gradient (dL/dy_pred)
        self.backward(grad_loss)                       # 4. Backpropagate and Accumulate Gradients

        return loss, y_pred

    def zero_grad(self):
        """Resets cumulative gradients in all layers. (1.2.3 zero_grad())"""
        for layer in self.layers:
            layer.zero_grad()

    def update(self, learning_rate, grad_accumulation_steps):
        """Updates all parameters and resets gradients. (1.2.3 update())"""
        for layer in self.layers:
            # We pass the accumulation steps to normalize the accumulated gradient
            layer.update(learning_rate, grad_accumulation_steps)

    def predict(self, x):
        """Performs a forward pass and returns the prediction. (1.2.3 predict(x))"""
        return self.forward(x)

    def save_to(self, path):
        """Saves all model parameters to a .npz file. (1.2.3 save_to)"""
        params = {}
        for i, layer in enumerate(self.layers):
            params[f'W{i}'] = layer.W
            params[f'b{i}'] = layer.b
        np.savez(path, **params)
        print(f"Model saved to {path}")

    def load_from(self, path):
        """Loads all model parameters from a .npz file. (1.2.3 load_from)"""
        data = np.load(path)

        # Architecture Check (1.2.3 - should throw error if shapes mismatch)
        for i, layer in enumerate(self.layers):
            W_shape = data[f'W{i}'].shape
            b_shape = data[f'b{i}'].shape

            if W_shape != layer.W.shape or b_shape != layer.b.shape:
                raise ValueError(
                    f"Architecture mismatch in layer {i}. "
                    f"Weights: Expected {layer.W.shape}, Got {W_shape}. "
                    f"Biases: Expected {layer.b.shape}, Got {b_shape}."
                )

        # Load Parameters
        for i, layer in enumerate(self.layers):
            layer.W = data[f'W{i}']
            layer.b = data[f'b{i}']
        print(f"Model loaded from {path}")

### 2.1 Setup and Visualization

In [ ]:
def load_and_preprocess_images(smiley_path, cat_path, target_size=256):
    """Loads, resizes, and normalizes the required images."""

    # --- Smiley (Grayscale) ---
    smiley_img = Image.open(smiley_path).convert('L') # 'L' for grayscale
    smiley_np = np.array(smiley_img, dtype=np.float32) / 255.0 # Normalize to [0, 1]

    # Ensure it's 256x256 (though specified as already being this size)
    if smiley_np.shape != (target_size, target_size):
        smiley_img = smiley_img.resize((target_size, target_size))
        smiley_np = np.array(smiley_img, dtype=np.float32) / 255.0

    # --- Cat (RGB) ---
    cat_img = Image.open(cat_path).convert('RGB')
    cat_img = cat_img.resize((target_size, target_size)) # Resize required by spec
    cat_np = np.array(cat_img, dtype=np.float32) / 255.0 # Normalize to [0, 1]

    # The cat output shape will be (256, 256, 3)

    print(f"Smiley shape: {smiley_np.shape}, Channels: 1")
    print(f"Cat shape: {cat_np.shape}, Channels: 3")

    return smiley_np, cat_np

def visualize_targets(smiley_np, cat_np, username):
    """Displays the target images with required labels."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # Smiley Plot
    axes[0].imshow(smiley_np, cmap='gray')
    axes[0].set_title(f"Target: Smiley (Grayscale)")
    axes[0].set_xlabel("X Coordinate")
    axes[0].set_ylabel("Y Coordinate")
    axes[0].axis('off')

    # Cat Plot
    axes[1].imshow(cat_np)
    axes[1].set_title(f"Target: Cat (RGB)")
    axes[1].set_xlabel("X Coordinate")
    axes[1].set_ylabel("Y Coordinate")
    axes[1].axis('off')

    add_username_to_plot(username)

    plt.tight_layout()
    plt.show()

In [ ]:
smiley, cat = load_and_preprocess_images("data/smiley.png", "data/cat.jpg")
visualize_targets(smiley, cat, "gracy.garg")

### 2.2 Feature Expansions

In [ ]:
# Raw Input Feature Function
def get_raw(coords):
    """
    Transforms coordinates v=(x, y) into the raw input feature vector.
    γraw(x, y) = (x, y)

    Args:
        coords (np.array): Array of normalized (x, y) coordinates (N x 2).

    Returns:
        np.array: The coordinates themselves (N x 2).
    """
    return coords

In [ ]:
def get_polynomial(coords, order=5):
    """
    Expands coordinates into a polynomial basis up to the specified order,
    including all cross-terms x^i * y^j where i + j <= order.

    Args:
        coords (np.array): Array of normalized (x, y) coordinates (N x 2).
        order (int): The highest total polynomial power (k).

    Returns:
        np.array: The expanded feature matrix (N x M).
    """
    x = coords[:, 0:1] # N x 1
    y = coords[:, 1:2] # N x 1

    features = []

    # Iterate through all orders from 1 up to 'order' (k)
    for k in range(1, order + 1):
        # Iterate through all powers of x (i) from k down to 0
        for i in range(k, -1, -1):
            j = k - i # The power of y (j) is always k - i

            # The order 1 terms (x^1, y^1) are handled when k=1.
            # We exclude the order 0 term (constant 1) and avoid duplicating order 1 terms.
            if k == 1:
                # Only include x and y (i=1, j=0) and (i=0, j=1)
                if i == 1:
                    features.append(x)
                elif j == 1:
                    features.append(y)
            else:
                # For orders k >= 2, include the term x^i * y^j
                term = (x**i) * (y**j)
                features.append(term)

    return np.concatenate(features, axis=1)

In [ ]:
def get_fourier(coords, freq=10):
    """
    Maps coordinates into a sinusoidal embedding (Fourier features).

    Args:
        coords (np.array): Array of normalized (x, y) coordinates (N x 2).
        freq (int): The number of frequency components (k).

    Returns:
        np.array: The expanded feature matrix (N x (1 + 4 * k)).
    """
    x = coords[:, 0:1] # N x 1
    y = coords[:, 1:2] # N x 1

    # Create the frequency vector f_1 to f_k
    # Typically, these are powers of 2 (2^0, 2^1, ...) or linearly spaced,
    # but here we use a simple linear integer sequence 1 to freq.
    f_k = np.arange(1, freq + 1)

    # Basis functions (2 * pi * f_k * x) and (2 * pi * f_k * y)
    B_x = 2.0 * np.pi * f_k * x
    B_y = 2.0 * np.pi * f_k * y

    # Features start with the constant 1
    features = [np.ones_like(x)] # [1]

    # Sin/Cos terms for x
    features.append(np.sin(B_x)) # sin(2πf₁x), ..., sin(2πfₖx)
    features.append(np.cos(B_x)) # cos(2πf₁x), ..., cos(2πfₖx)

    # Sin/Cos terms for y
    features.append(np.sin(B_y)) # sin(2πf₁y), ..., sin(2πfₖy)
    features.append(np.cos(B_y)) # cos(2πf₁y), ..., cos(2πfₖy)

    # Resulting feature dimension is 1 (bias) + 2*freq (for x) + 2*freq (for y)
    return np.concatenate(features, axis=1)

### 2.3 Normalization and Modular MLP

In [ ]:
class Modular_Dataloader:
    """
    Handles data loading, coordinate generation, feature expansion (Raw/Poly/Fourier),
    and feature normalization for image reconstruction.
    """
    def __init__(self, img_path, image_type, method, order=3, freq=10, target_size=256):
        self.img_path = img_path
        self.image_type = image_type
        self.method = method
        self.order = order
        self.freq = freq
        self.target_size = target_size

        # 1. Load and prepare data/coordinates
        self.target_image, self.coords = self._load_and_generate_coords()

        # 2. Apply feature mapping and normalization
        self.features = self._apply_feature_mapping_and_normalize()

        # Final set of samples: [(feature_vector, pixel_value), ...]
        self.data_samples = self._prepare_samples()


    def _load_and_generate_coords(self):
        """
        Loads the image and generates normalized coordinates [0, 1].
        """
        try:
            # Load and resize the image
            img = Image.open(self.img_path).convert('L' if self.image_type == 'Gray' else 'RGB')
            img = img.resize((self.target_size, self.target_size))

            # Normalize pixel intensities (Y targets) to [0, 1]
            target_image = np.array(img, dtype=np.float32) / 255.0

        except FileNotFoundError:
            raise FileNotFoundError(f"Image not found at {self.img_path}")

        H, W = target_image.shape[:2]

        coords = []
        # Normalization factor to map 0 to 0 and W-1 to 1
        norm_factor_w = W - 1
        norm_factor_h = H - 1

        for i in range(H): # y-coordinate
            for j in range(W): # x-coordinate
                # Normalized coordinates (x, y) in range [0, 1]
                normalized_x = j / norm_factor_w
                normalized_y = i / norm_factor_h
                coords.append([normalized_x, normalized_y])

        return target_image, np.array(coords, dtype=np.float32)


    def _apply_feature_mapping_and_normalize(self):
        """
        Applies the specified feature transformation (gamma) and performs normalization
        as required by the task guidelines.
        """
        coords = self.coords
        features = None

        if self.method == "Raw":
            # Raw features: Already in [0, 1] range from coordinate generation.
            features = get_raw(coords)

        elif self.method == "Polynomial":
            features = get_polynomial(coords, order=self.order)

            # Polynomial Normalization: Rescale to avoid exploding values.
            # Find the max magnitude across the entire feature matrix and scale by it.
            # Note: We skip the first two columns (x, y) if they are to be kept at [0, 1].

            # Simple rescale: Find max value (V_max) and scale all terms > 1.
            V_max = np.max(np.abs(features))
            if V_max > 1.0:
                 # Scale all terms down relative to the maximum magnitude
                features = features / V_max

        elif self.method == "Fourier":
            # Fourier features: Input coordinates should be scaled to cover one full cycle.
            # We assume the input coordinates [0, 1] are already suitable for
            # the internal 2*pi * f * x calculation.
            features = get_fourier(coords, freq=self.freq)

            # No normalization is required here as sin/cos are already in [-1, 1].

        else:
            raise ValueError(f"Unknown feature mapping method: {self.method}")

        return features

    def _prepare_samples(self):
        """Pairs the features with the corresponding pixel values."""
        samples = []
        H, W = self.target_image.shape[:2]
        is_gray = self.image_type == 'Gray'

        # The coordinates are ordered row-by-row (i then j)
        for i in range(H):
            for j in range(W):
                index = i * W + j
                feature_vector = self.features[index]

                # Extract target pixel value (L or R,G,B)
                if is_gray:
                    pixel_value = np.array([self.target_image[i, j]]) # (1,) array
                else:
                    pixel_value = self.target_image[i, j] # (3,) array

                samples.append((feature_vector, pixel_value))

        return samples

    def get_shuffled_data(self):
        """Returns the complete dataset shuffled for training."""
        shuffled = self.data_samples.copy()
        np.random.shuffle(shuffled)
        return shuffled

    def get_feature_dim(self):
        """Returns the dimension of the output feature vector (M)."""
        return self.features.shape[1]

### 2.4 Training and Comparison

In [ ]:
# The constant MLP architecture (hidden sizes [64, 128, 128])
HIDDEN_SIZES = [128, 64, 32]
HIDDEN_SIZES_2 = [64, 128, 128]
COMMON_HPARAMS = {'learning_rate': 0.001, 'batch_size': 256} # Use a moderate LR/BS

def setup_mlp(input_dim, output_dim, hidden_sizes):
    """Creates the standard MLP architecture with Identity output and MSELoss."""
    layers = []
    current_input = input_dim

    for size in hidden_sizes:
        layers.append(LinearLayer(current_input, size, Tanh())) # Use Tanh for hidden layers
        current_input = size

    # Final layer uses Identity activation for normalized pixel values [0, 1]
    layers.append(LinearLayer(current_input, output_dim, Sigmoid()))

    return Model(layers, loss_fn_type='bce' if output_dim == 1 else 'mse')

def reconstruct_image(model, dataloader):
    """Uses the model to predict all pixels and reconstructs the image."""
    N_pixels = len(dataloader.data_samples)

    # Extract features in original order (not shuffled)
    X_features = np.array([s[0] for s in dataloader.data_samples])

    # Predict and clip output to valid intensity range [0, 1]
    y_pred = model.predict(X_features)
    y_pred_clipped = np.clip(y_pred, 0.0, 1.0)

    H, W = dataloader.target_size, dataloader.target_size
    output_channels = dataloader.target_image.shape[-1] if dataloader.image_type != 'Gray' else 1

    # Reshape the 1D prediction back into H x W x C (or H x W)
    if output_channels == 1:
        reconstructed_img = y_pred_clipped.reshape(H, W)
    else:
        reconstructed_img = y_pred_clipped.reshape(H, W, output_channels)

    return reconstructed_img, y_pred, N_pixels

In [ ]:
def train_reconstruction_model(model, dataloader, epochs, hparams, run_id, save_images=False):
    """Trains the model and saves the reconstructed image after each epoch."""

    lr = hparams['learning_rate']
    bs = hparams['batch_size']

    history_loss_epoch = []

    # Create unique run folder for image saving
    run_folder = f"reconstruction/{run_id}"
    if save_images:
        os.makedirs(run_folder, exist_ok=True)

    start_time = time.time()

    for epoch in tqdm(range(1, epochs + 1), desc=f"Training {run_id}"):

        # Get shuffled data
        data = dataloader.get_shuffled_data()
        num_batches = math.ceil(len(data) / bs)
        total_epoch_loss = 0.0

        for batch_idx in range(num_batches):
            start_idx = batch_idx * bs
            end_idx = min((batch_idx + 1) * bs, len(data))
            batch = data[start_idx:end_idx]

            x_batch = np.array([s[0] for s in batch])
            y_batch = np.array([s[1] for s in batch]).reshape(-1, model.layers[-1].output_width)

            # Train (Forward, Loss, Backprop, Accumulate)
            current_loss, _ = model.train(x_batch, y_batch)
            model.update(lr, grad_accumulation_steps=1) # Update every batch
            total_epoch_loss += current_loss * len(batch)

        avg_epoch_loss = total_epoch_loss / len(data)
        history_loss_epoch.append(avg_epoch_loss)

        # Save image at each epoch for GIF visualization
        if save_images:
            reconstructed_img, _, _ = reconstruct_image(model, dataloader)
            # Convert to 8-bit image for PIL saving
            img_to_save = (reconstructed_img * 255).astype(np.uint8)

            if img_to_save.ndim == 2: # Grayscale
                 pil_img = Image.fromarray(img_to_save, 'L')
            else: # RGB
                 pil_img = Image.fromarray(img_to_save, 'RGB')

            pil_img.save(os.path.join(run_folder, f"epoch_{epoch:03d}.png"))

    end_time = time.time()
    total_time = end_time - start_time

    # Final reconstruction and loss
    final_img, y_pred_final, N_pixels = reconstruct_image(model, dataloader)
    final_loss = model.loss_fn.compute(y_pred_final, np.array([s[1] for s in dataloader.data_samples]))

    return final_loss, total_time, model.layers[0].input_width, final_img, history_loss_epoch

In [ ]:
def create_gif_visualization(run_id_raw, run_id_poly, run_id_fourier, 
                             raw_loss_history, poly_loss_history, fourier_loss_history, 
                             image_name, epochs):
    """
    Loads saved epoch images from three different feature mapping runs, 
    stitches them into a 1x3 frame, adds a loss curve visualization 
    to each subplot, and saves the sequence as a GIF.
    """
    
    BASE_DIR = "reconstruction"
    raw_folder = os.path.join(BASE_DIR, run_id_raw)
    poly_folder = os.path.join(BASE_DIR, run_id_poly)
    fourier_folder = os.path.join(BASE_DIR, run_id_fourier)
    
    output_filename = f"{image_name}_feature_comparison.gif"
    
    frame_files = [] # List to hold all image objects (frames)
    
    # 1. Determine common image size (use the first image from Raw run)
    try:
        sample_img_path = os.path.join(raw_folder, "epoch_001.png")
        sample_img = Image.open(sample_img_path)
        W, H = sample_img.size
        
        # Calculate loss range for proper scaling
        all_losses = raw_loss_history + poly_loss_history + fourier_loss_history
        max_loss = max(all_losses) if all_losses else 1.0
        min_loss = min(all_losses) if all_losses else 0.0
        loss_range = max_loss - min_loss if max_loss != min_loss else 1.0
        
        # --- Pre-calculate Loss Plot Data ---
        try:
            font = ImageFont.truetype("arial.ttf", 16)
        except IOError:
            font = ImageFont.load_default()

    except FileNotFoundError:
        print("ERROR: Sample image not found. Ensure models are trained and images saved.")
        return

    # 2. Iterate through all epochs and stitch frames
    for epoch in range(1, epochs + 1):
        try:
            # Paths to the three images for the current epoch
            epoch_str = f"epoch_{epoch:03d}.png"
            img_raw = Image.open(os.path.join(raw_folder, epoch_str))
            img_poly = Image.open(os.path.join(poly_folder, epoch_str))
            img_fourier = Image.open(os.path.join(fourier_folder, epoch_str))
            
            # Create a new blank image for the 1x3 composite frame with space for plots
            plot_height = 80
            composite_frame = Image.new('RGB', (W * 3, H + 40 + plot_height), color='black')
            
            # Paste images side-by-side
            composite_frame.paste(img_raw, (0, 40))
            composite_frame.paste(img_poly, (W, 40))
            composite_frame.paste(img_fourier, (W * 2, 40))
            
            draw = ImageDraw.Draw(composite_frame)
            
            # 3. Add Legend/Loss Curve (Text Overlay)
            title_y = 5
            
            # Helper to format loss value safely
            def get_loss_string(history, e):
                 try:
                     # Subtract 1 from epoch since lists are 0-indexed
                     loss = history[e - 1] 
                     return f"Loss: {loss:.4f}"
                 except IndexError:
                     return "N/A"

            # Raw Features Label
            draw.text((W * 0 + 5, title_y), 
                      f"Raw ({get_loss_string(raw_loss_history, epoch)})", 
                      fill="white", font=font)
            
            # Polynomial Features Label
            draw.text((W * 1 + 5, title_y), 
                      f"Polynomial ({get_loss_string(poly_loss_history, epoch)})", 
                      fill="white", font=font)

            # Fourier Features Label
            draw.text((W * 2 + 5, title_y), 
                      f"Fourier ({get_loss_string(fourier_loss_history, epoch)})", 
                      fill="white", font=font)

            # 4. Draw loss curves at the bottom
            plot_y_start = H + 45
            plot_width = W - 20
            plot_x_margin = 10
            
            def draw_loss_curve(x_offset, history, color, label, current_epoch):
                if len(history) <= 1:
                    return
                    
                # Draw background for this plot
                plot_x = x_offset + plot_x_margin
                draw.rectangle([plot_x, plot_y_start, plot_x + plot_width, plot_y_start + plot_height - 20], 
                              outline='gray', fill='black')
                
                # Draw loss curve up to current epoch
                for i in range(1, min(current_epoch, len(history))):
                    # Calculate x coordinates
                    x1 = plot_x + (i - 1) * plot_width / max(1, len(history) - 1)
                    x2 = plot_x + i * plot_width / max(1, len(history) - 1)
                    
                    # Calculate y coordinates (normalized and inverted)
                    y1 = plot_y_start + plot_height - 20 - ((history[i-1] - min_loss) / loss_range) * (plot_height - 20)
                    y2 = plot_y_start + plot_height - 20 - ((history[i] - min_loss) / loss_range) * (plot_height - 20)
                    
                    # Clamp y values to plot bounds
                    y1 = max(plot_y_start, min(plot_y_start + plot_height - 20, y1))
                    y2 = max(plot_y_start, min(plot_y_start + plot_height - 20, y2))
                    
                    draw.line([(x1, y1), (x2, y2)], fill=color, width=2)

            # Draw the three loss plots
            draw_loss_curve(0, raw_loss_history, "red", "Raw", epoch)
            draw_loss_curve(W, poly_loss_history, "green", "Polynomial", epoch) 
            draw_loss_curve(W * 2, fourier_loss_history, "blue", "Fourier", epoch)

            # IMPORTANT: Add the frame AFTER creating it completely
            frame_files.append(composite_frame)
            
        except FileNotFoundError as e:
            print(f"Warning: Could not find image for epoch {epoch}: {e}")
            continue

    # 5. Save the frames as a GIF (OUTSIDE the epoch loop)
    if frame_files:
        frame_files[0].save(output_filename, 
                            save_all=True, 
                            append_images=frame_files[1:], 
                            duration=150, # Slightly slower for better readability
                            loop=0) # Loop forever (0)
        print(f"\nGIF saved successfully as {output_filename}")
        print(f"GIF contains {len(frame_files)} frames with loss curve visualization")
    else:
        print("\nERROR: No frames were generated for the GIF.")
        print("Checking if image files exist...")
        # Debug: Check if files exist
        for epoch in range(1, min(6, epochs + 1)):  # Check first 5 epochs
            epoch_str = f"epoch_{epoch:03d}.png"
            raw_path = os.path.join(raw_folder, epoch_str)
            poly_path = os.path.join(poly_folder, epoch_str)
            fourier_path = os.path.join(fourier_folder, epoch_str)
            
            print(f"Epoch {epoch}:")
            print(f"  Raw: {os.path.exists(raw_path)} - {raw_path}")
            print(f"  Poly: {os.path.exists(poly_path)} - {poly_path}")
            print(f"  Fourier: {os.path.exists(fourier_path)} - {fourier_path}")

In [ ]:
def execute_comparison_runs(image_name, img_path, image_type, epochs):
    """
    Executes the three feature mapping methods (Raw, Polynomial, Fourier)
    for a given image, collects results, and generates a comparison GIF.
    Args:
        image_name (str): Name of the image (e.g., 'smiley').
        img_path (str): Path to the image file.
        image_type (str): 'Gray' or 'RGB'.
        epochs (int): Number of training epochs.
    """
    # Determine output channels (1 for Gray, 3 for RGB)
    output_dim = 1 if image_type == 'Gray' else 3

    comparison_results = []

    # --- Variables to store data for the GIF Visualization ---
    run_id_poly_gif = None
    run_id_fourier_gif = None
    raw_history = None
    poly_history = None
    fourier_history = None
    # --------------------------------------------------------

    # --- 1. Raw Features (Baseline) ---
    dl_raw = Modular_Dataloader(img_path, image_type, 'Raw')
    input_dim_raw = dl_raw.get_feature_dim()
    run_id_raw = f"{image_name}_raw_baseline" # Define Raw run ID here

    model_raw = setup_mlp(input_dim_raw, output_dim, HIDDEN_SIZES)

    final_loss, total_time, input_dim, final_img, history = train_reconstruction_model(
        model_raw, dl_raw, epochs, COMMON_HPARAMS, run_id_raw, save_images=True
    )
    comparison_results.append({
        'Method': 'Raw', 'Config': 'N/A', 'Image': image_name, 'Final Loss': final_loss,
        'Epoch Time': total_time / epochs, 'Input Params': input_dim
    })
    raw_history = history # Store Raw history


    # --- 2. Polynomial Features (Orders 5, 15, 25) ---
    for order in [5, 15, 25]:
        run_id = f"{image_name}_poly_k{order}"
        dl = Modular_Dataloader(img_path, image_type, 'Polynomial', order=order)
        input_dim = dl.get_feature_dim()

        model = setup_mlp(input_dim, output_dim, HIDDEN_SIZES)
        save_flag = True
        final_loss, total_time, _, final_img, history = train_reconstruction_model(
            model, dl, epochs, COMMON_HPARAMS, run_id, save_images=save_flag
        )
        comparison_results.append({
            'Method': 'Polynomial', 'Config': f'k={order}', 'Image': image_name, 'Final Loss': final_loss,
            'Epoch Time': total_time / epochs, 'Input Params': input_dim
        })
        if order == 15:
            poly_history = history
            run_id_poly_gif = run_id # Store the specific run ID for the GIF


    # --- 3. Fourier Features (Frequencies 5, 15, 25) ---
    for freq in [5, 15, 25]:
        run_id = f"{image_name}_fourier_f{freq}"
        dl = Modular_Dataloader(img_path, image_type, 'Fourier', freq=freq)
        input_dim = dl.get_feature_dim()

        model = setup_mlp(input_dim, output_dim, HIDDEN_SIZES)
        save_flag = True

        final_loss, total_time, _, final_img, history = train_reconstruction_model(
            model, dl, epochs, COMMON_HPARAMS, run_id, save_images=save_flag
        )
        comparison_results.append({
            'Method': 'Fourier', 'Config': f'f={freq}', 'Image': image_name, 'Final Loss': final_loss,
            'Epoch Time': total_time / epochs, 'Input Params': input_dim
        })
        if freq == 15:
            fourier_history = history
            run_id_fourier_gif = run_id # Store the specific run ID for the GIF


    # 4. Tabulation of Results
    df = pd.DataFrame(comparison_results)

    # Format and print the table
    df['Final Loss'] = df['Final Loss'].apply(lambda x: f'{x:.4e}')
    df['Epoch Time'] = df['Epoch Time'].apply(lambda x: f'{x:.4f}s')

    print(f"\n--- Results for {image_name.upper()} ---")
    print(df[['Method', 'Config', 'Input Params', 'Final Loss', 'Epoch Time']])

    # 5. GIF Visualization
    # Now call the GIF function using the explicitly saved run IDs and history lists
    create_gif_visualization(
        run_id_raw=run_id_raw,
        run_id_poly=run_id_poly_gif,
        run_id_fourier=run_id_fourier_gif,
        raw_loss_history=raw_history,
        poly_loss_history=poly_history,
        fourier_loss_history=fourier_history,
        image_name=image_name,
        epochs=epochs
    )

    # Return all necessary data
    # return comparison_results, raw_history, poly_history, fourier_history

### Function Calls for Smiley and Cat (Commented Out to Prevent Long Execution Time)

In [ ]:
execute_comparison_runs("smiley", "data/smiley.png", "Gray", epochs=50)

In [ ]:
execute_comparison_runs("cat", "data/cat.jpg", "RGB", epochs=150)

### 2.5 Reconstruction on Blurred Images

In [ ]:
import numpy as np
import os
import time
from tqdm import tqdm
import math

def train_model_with_early_stopping(model, dataloader, run_name, method, hparams):
    """
    Trains the reconstruction model until convergence using early stopping.
    This function is adapted from the train_model in Section 1.2.

    Args:
        model: The initialized MLP model.
        dataloader: The Modular_Dataloader instance.
        run_name (str): Identifier for the run (e.g., 'blur_5').
        method (str): Feature mapping method ('Raw' or 'Fourier').
        hparams (dict): Dictionary containing LR, BS, epochs, patience, etc.

    Returns:
        float: The final average epoch loss upon convergence.
        float: Total time taken for training.
        int: Total samples seen.
        str: The unique run folder path.
        list: List of epoch loss history.
    """
    # Hyperparameters
    epochs = hparams.get('epochs', 100)
    learning_rate = hparams.get('learning_rate', 0.05)
    batch_size = hparams.get('batch_size', 256)
    patience = hparams.get('patience', 10)
    rel_loss_threshold = hparams.get('relative_loss_threshold', 0.01)

    epoch_losses = []

    # Logging setup (Create unique folder for this run)
    run_folder = f"runs/blur_experiment/{run_name}_{method}_{time.strftime('%H%M%S')}"
    os.makedirs(run_folder, exist_ok=True)

    total_samples_seen = 0
    samples_in_current_accumulation = 0 # Assuming GAS=1 for simplicity here

    start_time = time.time()

    # Main Training Loop
    for epoch in range(1, epochs + 1):
        data = dataloader.get_shuffled_data() # Use get_shuffled_data from Modular_Dataloader
        num_batches = math.ceil(len(data) / batch_size)
        total_epoch_loss = 0.0

        pbar = tqdm(range(num_batches), desc=f"{run_name} | Epoch {epoch}/{epochs}", leave=False)

        for batch_idx in pbar:
            start_idx = batch_idx * batch_size
            end_idx = min((batch_idx + 1) * batch_size, len(data))
            batch = data[start_idx:end_idx]

            # Prepare batch data (Features, Targets)
            x_batch = np.array([s[0] for s in batch])
            y_batch = np.array([s[1] for s in batch]).reshape(-1, model.layers[-1].output_width)

            # Train step
            current_loss, _ = model.train(x_batch, y_batch)
            model.update(learning_rate, grad_accumulation_steps=1) # Assuming GAS=1

            total_epoch_loss += current_loss * len(batch)
            total_samples_seen += len(batch)

            pbar.set_postfix({'loss': f'{current_loss:.4f}'})

        # End of Epoch
        avg_epoch_loss = total_epoch_loss / len(data)
        epoch_losses.append(avg_epoch_loss)

        # Early Stopping Check (1.2.5)
        if len(epoch_losses) >= patience:
            L_t = avg_epoch_loss
            L_t_minus_patience = epoch_losses[-(patience)]

            # Stop if L_t >= 0.99 * L_t-10
            if L_t >= (1 - rel_loss_threshold) * L_t_minus_patience:
                # print(f"Early stopping triggered at epoch {epoch}. Loss did not improve...")
                break

    total_time = time.time() - start_time

    # Final loss is the last recorded epoch loss
    final_loss = epoch_losses[-1] if epoch_losses else 0.0

    return final_loss, total_time, total_samples_seen, run_folder, epoch_losses

In [ ]:
BLUR_LEVELS = np.arange(0, 10) # Sigma = 0, 1, ..., 10
MAX_EPOCHS = 100
HIDDEN_SIZES = [64, 128, 128]
COMMON_HPARAMS = {'learning_rate': 0.05, 'batch_size': 256}

# Early stopping parameters from 1.2.5
EARLY_STOP_HPARAMS = {
    'epochs': MAX_EPOCHS,
    'patience': 10,
    'relative_loss_threshold': 0.01
}

# Combine common and early stopping parameters for train_model
TRAINING_HPARAMS = {**COMMON_HPARAMS, **EARLY_STOP_HPARAMS}

comparison_data = []
username = "gracy.garg"

def run_blur_experiment():
    for sigma in BLUR_LEVELS:
        img_path = f"data/blurred/blur_{sigma}.png" # Assumes files are named blur_0.jpg, etc.
        image_name = f"blur_{sigma}"
        output_dim = 3 # RGB image

        # --- Method 1: BASE (Raw Coordinates) ---
        method = 'Raw'
        dl_raw = Modular_Dataloader(img_path, 'RGB', method)
        model_raw = setup_mlp(dl_raw.get_feature_dim(), output_dim, HIDDEN_SIZES)

        # Train with Early Stopping
        final_loss_raw, _, _, folder_raw, _ = train_model_with_early_stopping(
            model_raw, dl_raw, image_name, method, TRAINING_HPARAMS
        )

        comparison_data.append({
            'Sigma': sigma, 'Method': 'BASE (Raw)', 'Final Loss': final_loss_raw
        })

        # --- Method 2: FOURIER (Frequency k=5) ---
        method = 'Fourier'
        freq_k = 5
        dl_fourier = Modular_Dataloader(img_path, 'RGB', method, freq=freq_k)
        model_fourier = setup_mlp(dl_fourier.get_feature_dim(), output_dim, HIDDEN_SIZES)

        # Train with Early Stopping
        final_loss_fourier, _, _, folder_fourier, _ = train_model_with_early_stopping(
            model_fourier, dl_fourier, image_name, method, TRAINING_HPARAMS
        )

        comparison_data.append({
            'Sigma': sigma, 'Method': 'FOURIER (k=5)', 'Final Loss': final_loss_fourier
        })

        # Generate final reconstructed images for visual inspection
        final_img_raw, _, _ = reconstruct_image(model_raw, dl_raw)
        final_img_fourier, _, _ = reconstruct_image(model_fourier, dl_fourier)

        # Save reconstructed images for visual comparison
        os.makedirs("reconstructed", exist_ok=True)

        # Convert to 8-bit and save
        img_raw_8bit = (np.clip(final_img_raw, 0, 1) * 255).astype(np.uint8)
        img_fourier_8bit = (np.clip(final_img_fourier, 0, 1) * 255).astype(np.uint8)

        Image.fromarray(img_raw_8bit, 'RGB').save(f"reconstructed/blur_{sigma}_raw.png")
        Image.fromarray(img_fourier_8bit, 'RGB').save(f"reconstructed/blur_{sigma}_fourier.png")

    # Convert to DataFrame for easy visualization
    return pd.DataFrame(comparison_data)

In [ ]:
def visualize_blur_comparison(df_results, username):

    # Pivot the table to easily plot two series
    df_pivot = df_results.pivot(index='Sigma', columns='Method', values='Final Loss')
    raw_loss = df_pivot['BASE (Raw)']
    fourier_loss = df_pivot['FOURIER (k=5)']

    # --- Plot 1: Linear Scale ---
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)

    plt.plot(df_pivot.index, raw_loss, marker='o', label='BASE (Raw)')
    plt.plot(df_pivot.index, fourier_loss, marker='x', label='FOURIER (k=5)')

    plt.title("Final Reconstruction Loss vs. Blur Level (Linear Scale)")
    plt.xlabel(r"Gaussian Blur Level ($\sigma$)")
    plt.ylabel("Reconstruction Loss (MSE)")
    plt.legend()
    plt.grid(True)

    # --- Plot 2: Logarithmic Scale ---
    plt.subplot(1, 2, 2)

    plt.plot(df_pivot.index, raw_loss, marker='o', label='BASE (Raw)')
    plt.plot(df_pivot.index, fourier_loss, marker='x', label='FOURIER (k=5)')

    plt.yscale('log') # CRITICAL: Logarithmic Y-axis

    plt.title("Final Reconstruction Loss vs. Blur Level (Log Scale)")
    plt.xlabel(r"Gaussian Blur Level ($\sigma$)")
    plt.ylabel("Reconstruction Loss (MSE, Log Scale)")
    plt.legend()
    plt.grid(True, which="both", ls="-", alpha=0.3)

    # Add username to the figure
    plt.suptitle(f"Blur Reconstruction Comparison - User: {username}", y=1.02)
    plt.tight_layout()
    plt.savefig(f"blur_reconstruction_comparison.png")
    plt.show()

In [ ]:
# --- 1. Execute the Experiment and Collect Data ---
print("=====================================================================")
print("=== STARTING BLUR RECONSTRUCTION COMPARISON (Sigma 0 to 10) ===")
print("=====================================================================")

# Execute the experiment, which trains 22 models (2 methods * 11 blur levels)
# This call returns the pandas DataFrame containing all final loss values.
df_comparison_results = run_blur_experiment()

print("\n--- EXPERIMENT COMPLETE. RESULTS TABLE ---")
print(df_comparison_results)

# --- 2. Generate Required Visualizations ---
username = "gracy.garg" # IMPORTANT: Use your actual username

print("\n=== GENERATING FINAL COMPARISON PLOTS ===")

# Generate the two comparison plots (Linear and Log Y-axis)
visualize_blur_comparison(df_comparison_results, username)

print("\nExperiment successfully completed. Check the output folder for the saved plot.")

### Observation

* The graph strongly confirms the influence of **spectral bias** on image reconstruction. 

* At low blur ($\sigma=0$), the **FOURIER features are essential**, achieving a loss ($\approx 0.020$) less than half that of the BASE (Raw) model ($\approx 0.045$), which struggles severely to approximate the image's high-frequency sharp edges. 

* As the blur level increases, the target image transforms into a smooth, low-frequency function. The BASE MLP's bias towards learning smooth mappings becomes efficient, causing its loss to decrease steeply and **converge toward the FOURIER performance**.

* This illustrates that the high-frequency benefit of sinusoidal features diminishes when the target data lacks high-frequency content. 

### Visual Verification of Blur Reconstruction

The visual inspection of the reconstructed images provides crucial verification of the observed trends in the loss graph. This demonstrates the impact of high-frequency feature loss due to blurring.

1.  **Sharp Images ($\sigma=0$):**
    * **FOURIER Reconstruction:** The image appears significantly **sharper and more recognizable**. The Fourier features successfully encoded the high-frequency information (edges, textures) that define the image's fine details, leading to the lowest reconstruction loss at this level.
    * **BASE (Raw) Reconstruction:** The image is distinctly **blurry and smoothed**, lacking sharp boundaries. This visually confirms that the $\text{BASE}$ MLP struggled to capture the high-frequency content, resulting in the highest initial loss.

2.  **Blurred Images ($\sigma=5$ to $\sigma=9$):**
    * **Image Quality:** Both the **BASE** and **FOURIER** reconstructed images appear **visually similar** to the corresponding highly blurred original image.
    * **Reasoning:** The strong initial blur removes the challenging high-frequency information. Since the target function is now smooth, both MLPs efficiently reconstruct the low-frequency contours. This visual similarity confirms why the $\text{BASE}$ loss curve converges toward the $\text{FOURIER}$ loss curve at higher $\sigma$ values: the high-frequency advantage of the Fourier mapping is rendered irrelevant when the target data itself is smooth.